# Week 4 실습 — EDA, 상관관계와 관계 시각화

> "변수 A와 B의 상관계수가 높게 나왔습니다. 그러면 A가 B에 영향을 준다고 보고하면 될까요?" — 분석팀 인턴

**핵심 질문: 숫자 하나만 보고 변수의 관계를 판단해도 되는가?**

지난주에는 변수의 척도와 대표값을 선택했다. 이번 주에는 데이터를 먼저 탐색하고, 관계를 숫자로 확인한 뒤, 실제 분포를 그려 그 숫자를 다시 점검한다.

| Part | 내용 | 데이터 |
| --- | --- | --- |
| 0 | 도입 — 같은 요약 통계, 다른 데이터 | 교재 외 이해용 예시 |
| 1 | EDA — 데이터 상태 확인 | Hotel Booking Demand |
| 2 | 공분산 — 두 변수가 같이 움직인다는 것 | 작은 예시 |
| 3 | 상관계수 — 관계의 방향과 강도 | 작은 예시 |
| 4 | 상관계수만 보면 놓치는 것 | 교재 외 이해용 예시 |
| 5 | Wine Quality 상관성 분석 | Wine Quality |
| 6 | 관계 시각화 | 50 Startups |
| 7 | 최종 분석 미션 | Wine Quality |

**셀 표시**

- [실행]: 그대로 실행
- [빈칸]: 코드의 TODO를 직접 채움
- [예측]: 코드를 돌리기 전에 결과를 예상
- [해석]: 출력과 그래프를 보고 자신의 말로 설명

코드 문법과 오류는 LLM에 물어봐도 된다. 방법 선택과 그래프·통계량 해석은 [해석]에 자기 생각을 먼저 적은 뒤 확인한다. 발표용 정답을 보기 전에 본인 답을 남긴다.


## 기본 세팅 [실행]

세 데이터는 저장소의 data 폴더에 있다. 인터넷 다운로드는 하지 않는다. 노트북을 저장소 루트, week04 폴더, presenter 폴더 중 어디에서 실행해도 루트 폴더를 찾는다.

In [ ]:
# 확인할 것: PROJECT_ROOT 아래에 data와 common이 있고, Python 패키지 import가 끝나는지.
from pathlib import Path
import sys

_candidates = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (p for p in _candidates if (p / "data").is_dir() and (p / "common" / "load_data.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("DAstudy 저장소 안에서 노트북을 실행해 주세요.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.load_data import DATA_DIR
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager
from IPython.display import display

pd.set_option("display.float_format", "{:.3f}".format)
plt.rcParams["figure.dpi"] = 100
_installed = {f.name for f in font_manager.fontManager.ttflist}
for _font in ["AppleGothic", "Malgun Gothic", "NanumGothic", "Nanum Gothic", "Noto Sans CJK KR", "Noto Sans KR"]:
    if _font in _installed:
        plt.rcParams["font.family"] = _font
        break
plt.rcParams["axes.unicode_minus"] = False
print("데이터 폴더 확인:", DATA_DIR.is_dir())


---
## Part 0. 숫자만 보면 충분할까? [실행, 예측, 해석]

**교재 외 이해용 예시 — 앤스컴의 네 데이터(Anscombe's quartet).** 네 묶음의 평균·표준편차·상관계수를 먼저 보고, 산점도 모양을 예상한다. 이 도입은 뒤에서 배울 EDA와 관계 시각화가 필요한 이유를 보여준다.


In [ ]:
# 확인할 것: 네 묶음의 x/y 평균·표준편차·피어슨 상관이 거의 같은지.
anscombe = {
    "I": ([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II": ([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": ([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV": ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}
ans_rows = []
for name, (x, y) in anscombe.items():
    ans_rows.append((name, np.mean(x), np.mean(y), np.std(x, ddof=1), np.std(y, ddof=1), np.corrcoef(x, y)[0, 1]))
ans_summary = pd.DataFrame(ans_rows, columns=["묶음", "x 평균", "y 평균", "x 표준편차", "y 표준편차", "상관계수"]).set_index("묶음")
display(ans_summary.round(2))


[예측] 네 묶음의 산점도 모양도 비슷할까? 표만 보고 예상하고 이유를 적는다.

- 비슷할 것이라고 예상하기 쉽다. 요약 통계가 거의 같기 때문이다. 그래프를 보기 전에는 모양을 확정할 수 없다.

In [ ]:
# 확인할 것: 표의 숫자는 비슷하지만 직선·곡선·이상치 등 네 모양이 다른지.
fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True)
for ax, (name, (x, y)) in zip(axes.flat, anscombe.items()):
    ax.scatter(x, y, s=35, color="#2a78d6")
    ax.set_title(name)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


[해석] 상관계수만 전달받았다면 무엇을 놓쳤는가?

- II의 곡선, III·IV에서 특정 점이 전체 관계를 좌우하는 모습을 놓친다. 같은 숫자라도 실제 점의 배치는 다르다.

### Part 0 정리

> **발제자 포인트:** 표의 숫자가 같아도 데이터의 모양은 다르다. 이 Part에서는 원인을 모두 설명하지 말고 질문을 남긴다.
>
> **연결 문장:** 그렇다면 실제 데이터 분석에서는 숫자 계산 전에 무엇부터 확인해야 할까?

---
## Part 1. EDA — 데이터 상태 확인 [실행, 빈칸, 예측, 해석]

교재 10.1의 Hotel Booking Demand 실습 흐름을 따른다. 한 행은 호텔 예약 기록이다. 주요 변수는 호텔 종류, 취소 여부, 예약부터 투숙까지의 기간(lead_time), 투숙 연도, 아동 수, 기업 코드, 객실 요금(adr)이다. **예약 기록 수와 고유 고객 수는 다를 수 있다.**


In [ ]:
# 확인할 것: (119390, 32), hotel 두 범주, lead_time·children·company 열이 있는지.
hotel = pd.read_csv(DATA_DIR / "hotel_bookings.csv")
assert hotel.shape == (119390, 32), hotel.shape
print("Hotel Booking Demand:", hotel.shape)
display(hotel.head())


In [ ]:
# 확인할 것: dtype과 non-null 수, 특히 children과 company의 결측 규모.
hotel.info()
display(hotel[["children", "company", "lead_time", "arrival_date_year"]].isna().sum().to_frame("결측 개수"))


[해석] children·company의 결측과 arrival_date_year의 자료형을 보고 무엇을 주의해야 하는가?

- children 결측은 소수지만 company는 대부분 비어 있다. year가 정수로 저장됐다고 평균 연도를 연속형 수치처럼 해석하면 안 된다. 결측 이유는 변수 설명과 기록 과정을 더 확인해야 한다.

### 1-4. 숫자형 변수 요약 [빈칸]

교재의 describe 실습처럼 평균, 중앙값(50%), 최솟값, 최댓값을 나란히 본다.

In [ ]:
# 확인할 것: lead_time 평균과 중앙값, 최댓값과 75% 값의 간격을 비교한다.
hotel_summary = hotel.describe(include="number").T
display(hotel_summary.loc[["lead_time", "arrival_date_year", "children", "company", "adr"]])


[해석] 평균과 중앙값이 다른 변수, 숫자형이지만 단순 평균 해석이 곤란한 변수를 적는다.

- lead_time은 평균 약 104일, 중앙값 69일로 오른쪽 꼬리를 의심할 수 있다. arrival_date_year는 시기 구분이며 company는 기업 코드라 평균 자체가 분석 목적에 맞지 않는다.

### 1-5. 왜도·첨도와 분포 [예측, 실행, 해석]

왜도는 좌우 비대칭, 첨도는 극단값이 나오는 꼬리의 정도를 살펴보는 지표다. 숫자만으로 모양을 확정하지 않고 그래프를 함께 본다.

[예측] lead_time은 좌우대칭일까, 오른쪽 꼬리가 길까? describe 결과를 근거로 적는다.

- 평균이 중앙값보다 크고 최댓값이 멀리 있으므로 오른쪽 꼬리가 길 것으로 예상한다.

In [ ]:
# 확인할 것: lead_time의 왜도·첨도와 히스토그램의 오른쪽 꼬리가 같은 이야기를 하는지.
print("lead_time 왜도:", round(hotel["lead_time"].skew(), 3))
print("lead_time 첨도:", round(hotel["lead_time"].kurtosis(), 3))
print("lead_time=0 건수:", int((hotel["lead_time"] == 0).sum()))
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data=hotel, x="lead_time", bins=60, color="#2a78d6", ax=ax)
ax.set_title("Hotel booking lead_time distribution")
plt.tight_layout()
plt.show()


교재에는 distplot이 쓰이지만 최신 seaborn에서는 histplot을 사용한다. lead_time=0은 당일 예약일 수도 있고 기록 방식의 결과일 수도 있다. 그래프만으로 원인을 확정하지 않는다.

[해석] lead_time=0과 긴 오른쪽 꼬리를 어떻게 읽을 수 있는가? 추가로 확인할 사항은?

- 0일 기록은 약 6,345건이고 분포는 오른쪽으로 길다. 0이 실제 당일 예약인지 데이터 생성 규칙을 확인해야 하며, 평균만으로 전형적인 예약을 설명하기 어렵다.

### 1-7. 호텔 그룹별 lead_time [실행, 해석]

최댓값 하나보다 전체 분포를 비교한다. 그림 속 점은 가독성을 위해 호텔별 400건을 고정 seed로 추출한다. 분포 형태(violin)는 전체 데이터를 이용한다.

In [ ]:
# 확인할 것: 최댓값이 더 큰 호텔과 분포의 중심이 더 큰 호텔이 같은지.
hotel_order = ["Resort Hotel", "City Hotel"]
hotel_points = hotel.groupby("hotel", group_keys=False).sample(n=400, random_state=42)
display(hotel.groupby("hotel")["lead_time"].agg(["size", "mean", "median", "max"]).loc[hotel_order])
fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(data=hotel, x="hotel", y="lead_time", order=hotel_order, inner=None, color="#b8c9e6", ax=ax)
sns.stripplot(data=hotel_points, x="hotel", y="lead_time", order=hotel_order, size=2, alpha=0.35, color="#1f4e79", ax=ax)
ax.set_title("Lead time by hotel (points: 400 per group)")
plt.tight_layout()
plt.show()


[해석] 최댓값만 봤을 때와 violinplot을 봤을 때 결론이 어떻게 달라지는가?

- Resort Hotel의 최댓값이 더 크더라도 City Hotel의 전반적인 lead_time 분포가 더 긴 쪽에 놓인다. 한두 극단값 대신 중앙값과 분포 모양을 함께 봐야 한다.

[해석] EDA에서 발견한 주의사항 세 가지를 숫자 또는 열 이름과 함께 적는다.

- ① company 결측이 매우 많다. ② lead_time은 오른쪽 꼬리가 길고 0도 여러 건이다. ③ arrival_date_year·company처럼 숫자 저장 형태와 분석 척도가 다른 열이 있다.

### Part 1 정리

> **발제자 포인트:** EDA는 데이터 구조와 기록상의 이상을 먼저 발견하는 과정이다. 0이나 결측의 뜻은 맥락을 확인해야 한다.
>
> **연결 문장:** 이제 두 변수가 함께 움직이는 모습을 숫자로 나타내 보자.

---
## Part 2. 공분산 — 함께 움직이는 방향 [예측, 빈칸, 해석]

교재 10.2.1을 작은 데이터로 계산한다. 공부 시간과 점수를 예시로 쓰되, 이 표는 원리를 이해하기 위한 **교재 외 이해용 예시**다. 상관이나 인과에 대한 현실 결론으로 읽지 않는다.


In [ ]:
# 확인할 것: 관측값은 5개이며 두 열의 평균을 계산할 수 있는지.
tiny = pd.DataFrame({"study_hours": [1, 2, 3, 4, 5], "score": [50, 60, 65, 75, 85]})
display(tiny)


[예측] X와 Y가 둘 다 각자의 평균보다 크다면 (X-평균X)×(Y-평균Y)의 부호는?

- 양수다. 양수인 두 편차를 곱하기 때문이다.

In [ ]:
# 확인할 것: 편차 곱의 합과 표본 공분산의 부호를 확인한다.
x_mean = tiny["study_hours"].mean()
y_mean = tiny["score"].mean()
x_dev = tiny["study_hours"] - x_mean
y_dev = tiny["score"] - y_mean
dev_product = x_dev * y_dev
covariance = dev_product.sum() / (len(tiny) - 1)
cov_steps = tiny.assign(x_dev=x_dev, y_dev=y_dev, dev_product=dev_product)
display(cov_steps)
print("표본 공분산:", covariance)


In [ ]:
# 확인할 것: 직접 계산한 표본 공분산과 pandas·NumPy 결과가 같은지.
pandas_cov = tiny.cov().loc["study_hours", "score"]
numpy_cov = np.cov(tiny["study_hours"], tiny["score"], ddof=1)[0, 1]
assert np.isclose(covariance, pandas_cov)
assert np.isclose(covariance, numpy_cov)
print("직접 계산 / pandas / NumPy:", covariance, pandas_cov, numpy_cov)


[해석] 양의 공분산과 음의 공분산은 각각 무엇을 뜻하는가?

- 양수면 두 변수의 편차가 대체로 같은 방향이고, 음수면 반대 방향이다. 공분산 부호만으로 인과나 관계의 강도를 단정하지 않는다.

[예측] score를 10점 단위(score/10)로 바꾸면 공분산 값은 유지될까?

- 유지되지 않는다. 두 번째 변수의 단위가 10분의 1이 되면 공분산도 10분의 1이 된다.

In [ ]:
# 확인할 것: 값의 뜻은 같지만 공분산 숫자는 단위에 따라 바뀌는지.
tiny["score_10pt"] = tiny["score"] / 10
cov_scaled = tiny[["study_hours", "score_10pt"]].cov().iloc[0, 1]
display(pd.DataFrame({"표현": ["원래 점수", "10점 단위"], "공분산": [covariance, cov_scaled]}))
assert np.isclose(cov_scaled, covariance / 10)


[해석] 관계 자체는 같은데 공분산 숫자가 달라진 이유는?

- 공분산은 두 변수의 편차를 곱하므로 각 변수의 단위와 크기에 영향을 받는다.

### Part 2 정리

> **발제자 포인트:** 공분산의 부호는 함께 움직이는 방향을 보여주지만 절댓값은 단위에 좌우된다.
>
> **연결 문장:** 서로 다른 단위의 변수도 비교할 수 있는 척도가 있을까?

---
## Part 3. 상관계수 — 단위를 제거한 관계의 숫자 [빈칸, 해석]

교재 10.2.2의 피어슨 상관계수는 공분산을 두 변수의 표본 표준편차로 나눈다.

r = Cov(X, Y) / (sX × sY)

범위는 -1부터 1까지다. 부호는 방향, 절댓값은 **선형 관계**의 정도를 나타낸다. r이 0에 가까우면 뚜렷한 선형 관계가 확인되지 않는다는 뜻이지, 모든 종류의 관계가 없다는 뜻은 아니다.


In [ ]:
# 확인할 것: 직접 구한 r이 pandas corr() 값과 같은지, -1~1 범위인지.
sx = tiny["study_hours"].std(ddof=1)
sy = tiny["score"].std(ddof=1)
corr_xy = covariance / (sx * sy)
pandas_corr = tiny[["study_hours", "score"]].corr().iloc[0, 1]
assert np.isclose(corr_xy, pandas_corr)
assert -1 <= corr_xy <= 1
print("직접 계산 / pandas:", corr_xy, pandas_corr)


In [ ]:
# 확인할 것: score의 단위를 바꾸면 공분산은 변하지만 상관계수는 유지되는지.
corr_scaled = tiny[["study_hours", "score_10pt"]].corr().iloc[0, 1]
compare_units = pd.DataFrame({
    "데이터": ["원래 점수", "10점 단위"],
    "공분산": [covariance, cov_scaled],
    "상관계수": [corr_xy, corr_scaled],
})
display(compare_units)
assert np.isclose(corr_xy, corr_scaled)


[해석] 두 행에서 달라진 숫자와 유지된 숫자는 무엇인가?

- 공분산은 단위 변환에 따라 변했고, 상관계수는 그대로다. 표준편차로 나누면서 단위의 영향을 제거했기 때문이다.

**해석 주의:** 상관계수는 산점도의 기울기나 효과 크기 자체가 아니다. 예를 들어 R&D 지출과 수익이 함께 커져도, 지출을 늘리면 수익이 증가한다고 이 자료만으로 증명할 수 없다.

### Part 3 정리

> **발제자 포인트:** r의 부호와 절댓값을 구분해 읽는다. 0에 가까운 r은 선형 관계에 대한 정보다.
>
> **연결 문장:** 이 숫자가 작거나 클 때 실제 점의 모양은 언제나 예상대로일까?

---
## Part 4. 상관계수만 보면 놓치는 것 [예측, 실행, 해석]

**교재 외 이해용 예시.** 교재 10.2는 곡선 관계와 이상치가 상관계수의 해석을 바꿀 수 있다고 설명한다. 작은 예시로 숫자와 산점도를 나란히 확인한다.


[예측] y=x²인 대칭 데이터는 뚜렷한 관계가 있다. 피어슨 r도 크게 나올까?

- x가 음수에서 양수로 바뀌며 기울기 방향도 바뀌므로, 관계는 뚜렷해도 선형 상관은 거의 0일 것으로 예상한다.

In [ ]:
# 확인할 것: 곡선은 분명하지만 피어슨 r이 0에 가까운지.
curve_x = np.arange(-5, 6)
curve_y = curve_x ** 2
curve_r = np.corrcoef(curve_x, curve_y)[0, 1]
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(curve_x, curve_y, color="#2a78d6")
ax.set(xlabel="x", ylabel="x squared", title=f"Curved relation; Pearson r = {curve_r:.3f}")
plt.tight_layout()
plt.show()
print("피어슨 r:", round(curve_r, 3))


[해석] r≈0을 이 데이터에서 어떻게 정확히 말해야 하는가?

- 뚜렷한 선형 관계가 확인되지 않는다고 말할 수 있다. U자 형태의 비선형 관계는 존재한다.

[예측] 대체로 함께 증가하는 점들에 멀리 떨어진 점 하나를 추가하면 상관계수는 얼마나 달라질까?

- 한 점이 전체 계산에 큰 영향을 줄 수 있다. 방향까지 달라질 수도 있으므로 실제 점을 봐야 한다.

In [ ]:
# 확인할 것: 같은 기본 점에 극단값 한 점을 더했을 때 r과 산점도가 어떻게 바뀌는지.
base_x = np.arange(1, 11, dtype=float)
base_y = np.array([2, 3, 5, 4, 6, 7, 7, 9, 10, 11], dtype=float)
with_x = np.append(base_x, 20)
with_y = np.append(base_y, -5)
r_before = np.corrcoef(base_x, base_y)[0, 1]
r_after = np.corrcoef(with_x, with_y)[0, 1]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(base_x, base_y, color="#2a78d6")
axes[0].set_title(f"Before: r = {r_before:.3f}")
axes[1].scatter(with_x, with_y, color="#2a78d6")
axes[1].scatter([20], [-5], color="#eb6834", s=80)
axes[1].set_title(f"After: r = {r_after:.3f}")
for ax in axes:
    ax.set(xlabel="x", ylabel="y")
plt.tight_layout()
plt.show()


[해석] 극단값 추가 전후 r이 달라진 이유와 실제 분석에서 할 일을 적는다.

- 공분산과 표준편차 계산에 멀리 떨어진 점이 크게 기여했다. 실제 데이터라면 오류인지 진짜 관측값인지 확인하고, 포함·제외 결과를 함께 본다.

[해석] r=0.05라는 숫자만 전달받았다면 추가로 무엇을 확인할까?

- 산점도의 곡선·군집·이상치, 변수의 단위와 범위, 하위 집단별 관계를 확인한다.

### Part 4 정리

> **발제자 포인트:** 상관계수 한 개와 산점도 한 장을 같이 보여준다. 이상치 제거는 이유를 확인한 뒤 결정한다.
>
> **연결 문장:** 이제 교재의 실제 Wine Quality 데이터에서 여러 변수의 관계를 살펴보자.

---
## Part 5. Wine Quality 상관성 분석 [실행, 빈칸, 해석]

교재 10.2.3의 흐름을 따른다: 데이터 확인 → 산점도 행렬 → 공분산 행렬 → 상관행렬 → 히트맵 → 클러스터 히트맵. type은 범주형이므로 공분산·피어슨 상관에 넣지 않는다. 열마다 소량의 결측이 있으며, pandas는 변수쌍마다 사용 가능한 행으로 계산한다.


In [ ]:
# 확인할 것: (6497, 13)과 실제 열 이름, quality와 type의 자료형.
wine = pd.read_csv(DATA_DIR / "wine-quality.csv")
assert wine.shape == (6497, 13), wine.shape
print("Wine Quality:", wine.shape)
display(wine.head())
print(wine.dtypes.to_string())
display(wine.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).to_frame("결측 개수"))


전체 6,497행으로 12개 수치 열의 모든 쌍을 그리면 읽기 어렵다. 산점도 행렬은 주요 4개 열과 고정 seed 표본 400행으로 모양을 살피고, 뒤의 행렬 계산은 전체 데이터를 사용한다.

In [ ]:
# 확인할 것: 표본의 산점도에서 직선·군집·이상치를 살핀다. 표본은 그림용이다.
wine_plot = wine[["alcohol", "density", "volatile acidity", "quality"]].sample(n=400, random_state=42)
g = sns.pairplot(wine_plot, corner=True, diag_kind="hist", plot_kws={"alpha": 0.35, "s": 15})
g.fig.suptitle("Wine Quality pairplot (400-row sample)", y=1.02)
plt.show()


In [ ]:
# 확인할 것: 숫자형 열만 선택되어 type이 빠지는지, 공분산과 상관행렬 크기가 같은지.
wine_num = wine.select_dtypes(include="number")
wine_cov = wine_num.cov()
wine_corr = wine_num.corr(method="pearson")
print("수치형 열:", list(wine_num.columns))
print("공분산 / 상관행렬 크기:", wine_cov.shape, wine_corr.shape)
display(wine_cov.round(3))
display(wine_corr.round(3))


### 5-4. quality와의 상관 순서 [빈칸]

부호는 유지하고 정렬 순서만 절댓값으로 정한다. 이 순위는 후보를 찾는 도구이지 원인을 정하는 순위가 아니다.

In [ ]:
# 확인할 것: quality 자신을 제외하고 절댓값 기준으로 정렬했을 때 양·음 상관의 부호가 남는지.
quality_corr = wine_corr["quality"].drop("quality").sort_values(key=lambda s: s.abs(), ascending=False)
display(quality_corr.to_frame("Pearson r").round(3))


In [ ]:
# 확인할 것: 강한 양·음 관계의 위치가 숫자 행렬과 색으로 일치하는지.
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(wine_corr, cmap="RdYlBu_r", vmin=-1, vmax=1, center=0, annot=True, fmt=".2f", annot_kws={"size": 7}, ax=ax)
ax.set_title("Wine Quality correlation heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# 확인할 것: 상관 패턴이 비슷한 변수가 가까이 묶이는지. 색과 수치도 함께 본다.
g = sns.clustermap(wine_corr, cmap="RdYlBu_r", vmin=-1, vmax=1, center=0, annot=True, fmt=".2f", annot_kws={"size": 6}, figsize=(10, 9))
g.fig.suptitle("Wine Quality clustered correlation", y=1.02)
plt.show()


[해석] quality와 상대적으로 양의 상관이 큰 변수와 음의 상관이 큰 변수는?

- alcohol이 약 +0.444로 가장 큰 양의 상관, density가 약 -0.306으로 큰 음의 상관이다. volatile acidity도 약 -0.266이다. 수치가 작다고 관계가 전혀 없다고 단정하지 않는다.

[해석] 이 표에서 quality에 대한 인과효과를 말할 수 있는가? 입력변수끼리 강한 상관이 있으면 무엇을 기억할까?

- 관측 자료의 상관만으로 원인을 말할 수 없다. 입력변수끼리 강한 상관은 이후 회귀분석에서 다중공선성 점검이 필요하다는 단서다.

### Part 5 정리

> **발제자 포인트:** 상관행렬은 변수 후보를 빠르게 찾는 도구다. 그래프와 데이터 생성 과정을 함께 봐야 한다.
>
> **연결 문장:** 두 변수 관계를 청중이 한눈에 볼 수 있게 그리면 무엇이 더 드러날까?

---
## Part 6. 관계 시각화 [예측, 실행, 빈칸, 해석]

교재 10.6의 50 Startups 예제다. 한 행은 스타트업 한 곳이다. R&D·관리·마케팅 지출과 수익의 관계를 산점도, 투명도, 회귀선, 버블 차트로 차례로 본다.


In [ ]:
# 확인할 것: (50, 5), R&D Spend·Profit 등의 실제 열 이름과 결측 여부.
startups = pd.read_csv(DATA_DIR / "50_Startups.csv")
assert startups.shape == (50, 5), startups.shape
print("50 Startups:", startups.shape)
display(startups.head())
display(startups.isna().sum().to_frame("결측 개수"))


[예측] R&D Spend와 Profit의 산점도는 양·음·뚜렷한 선형 관계 없음 중 어디에 가까울까?

- 양의 선형 관계에 가까울 것으로 예상한다. 실제 산점도로 점의 퍼짐과 극단값도 확인해야 한다.

In [ ]:
# 확인할 것: 전체 방향과 멀리 떨어진 점의 위치를 본다.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(startups["R&D Spend"], startups["Profit"], s=50, color="#2a78d6")
ax.set(xlabel="R&D Spend", ylabel="Profit", title="R&D Spend vs Profit")
plt.tight_layout()
plt.show()


[해석] 산점도에서 방향·점의 퍼짐·극단값을 설명한다.

- 전반적으로 양의 선형 관계가 강하다. R&D 지출이 0에 가까운 기업들의 수익은 하나의 값으로 모이지 않는다.

In [ ]:
# 확인할 것: alpha를 낮추면 겹친 점을 구별하기 쉬워지는지.
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, alpha in zip(axes, [1.0, 0.3]):
    ax.scatter(startups["R&D Spend"], startups["Profit"], s=70, alpha=alpha, color="#2a78d6")
    ax.set(xlabel="R&D Spend", ylabel="Profit", title=f"alpha={alpha}")
plt.tight_layout()
plt.show()


[해석] 투명도 alpha가 점이 겹치는 그림에서 하는 일은?

- 겹친 점이 진하게 보여 밀집 구간을 알아보기 쉽게 한다. 다만 50개 표본에서는 차이가 크지 않을 수 있다.

In [ ]:
# 확인할 것: 회귀선이 산점도의 전체 방향을 요약하는지, 점이 선에서 얼마나 벗어나는지.
fig, ax = plt.subplots(figsize=(7, 4))
sns.regplot(data=startups, x="R&D Spend", y="Profit", scatter_kws={"alpha": 0.5, "s": 40}, line_kws={"color": "#eb6834"}, ax=ax)
ax.set_title("R&D Spend vs Profit with regression line")
plt.tight_layout()
plt.show()


회귀선은 관측된 관계의 방향을 보여준다. 선을 그렸다는 사실만으로 R&D 지출을 늘렸을 때 수익이 증가한다는 인과관계가 증명되지는 않는다.

### 6-4. 버블 차트 [빈칸]

가로축 R&D Spend, 세로축 Profit, 점의 면적 Marketing Spend, 색 Administration을 사용한다. Marketing Spend를 그림 크기 40~340 범위로 변환해 지나치게 큰 점을 막는다.

In [ ]:
# 확인할 것: x/y 외에 점 크기와 색이 다른 두 변수의 정보를 더하는지.
marketing = startups["Marketing Spend"]
bubble_size = 40 + 300 * (marketing - marketing.min()) / (marketing.max() - marketing.min())
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(
    x=startups["R&D Spend"],
    y=startups["Profit"],
    s=bubble_size,
    c=startups["Administration"],
    alpha=0.55,
    cmap="viridis",
    edgecolors="black",
    linewidths=0.3,
)
fig.colorbar(sc, ax=ax, label="Administration")
ax.set(xlabel="R&D Spend", ylabel="Profit", title="Size: Marketing Spend; color: Administration")
plt.tight_layout()
plt.show()


[해석] 일반 산점도보다 어떤 정보가 늘었는가? 마케팅 지출이 큰 기업은 항상 수익이 높은가? 관리비 패턴은 뚜렷한가?

- 점 크기로 마케팅 지출, 색으로 관리비가 추가됐다. 큰 점이 높은 수익에만 있지는 않고 관리비의 단순한 색 변화도 뚜렷하지 않다. 인과 결론은 낼 수 없다.

### Part 6 정리

> **발제자 포인트:** 관계 시각화는 숫자 한 개 뒤에 숨은 분포·극단값·겹침을 확인하는 단계다.
>
> **연결 문장:** 이제 숫자와 그래프를 함께 사용해 하나의 분석 결과를 정리해 보자.

---
## Part 7. 최종 분석 미션 [빈칸, 해석]

> 분석팀장: "Wine Quality에서 관심 변수 1~2개를 선택해 quality와 어떤 관계가 있는지 한 화면으로 설명해 주세요."

변수 선택 이유, 기술통계, 분포, 피어슨 상관, 산점도, 이상치, 비선형 가능성을 함께 보고한다. 변수 선택과 예상은 코드를 돌리기 전에 기록한다.


[예측] 선택할 변수 1~2개와 이유, quality와의 관계를 먼저 적는다.

- alcohol을 선택한다. 앞선 상관행렬에서 양의 상관이 상대적으로 컸고, 산점도에서도 양의 방향이 보일 것으로 예상한다.

In [ ]:
# 확인할 것: 변수의 분포·상관·IQR 바깥 건수와 quality 산점도를 한 화면에서 비교한다.
chosen_cols = ["alcohol"]  # 관심 수치 변수 1~2개를 선택한다.
assert 1 <= len(chosen_cols) <= 2
assert set(chosen_cols) <= set(wine_num.columns) - {"quality"}
final_rows = []
fig, axes = plt.subplots(len(chosen_cols), 2, figsize=(11, 4 * len(chosen_cols)), squeeze=False)
for i, col in enumerate(chosen_cols):
    series = wine[col].dropna()
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_n = int(((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum())
    r = wine[[col, "quality"]].corr().iloc[0, 1]
    final_rows.append((col, len(series), series.mean(), series.median(), series.min(), series.max(), r, outlier_n))
    sns.histplot(series, bins=35, ax=axes[i, 0], color="#2a78d6")
    axes[i, 0].set_title(f"{col} distribution")
    axes[i, 1].scatter(wine[col], wine["quality"], s=10, alpha=0.15, color="#eb6834")
    axes[i, 1].set(xlabel=col, ylabel="quality", title=f"{col} vs quality; r={r:.3f}")
plt.tight_layout()
plt.show()
final_table = pd.DataFrame(final_rows, columns=["변수", "유효 건수", "평균", "중앙값", "최솟값", "최댓값", "피어슨 r", "IQR 바깥 건수"]).set_index("변수")
display(final_table.round(3))


### 내가 말할 수 있는 것
- 이 자료에서 alcohol과 quality의 피어슨 상관은 약 +0.444다. 산점도에서도 전반적인 양의 방향이 보인다.

### 이 결과만으로 말할 수 없는 것
- 알코올 도수를 올리면 품질 점수가 높아진다는 인과효과는 말할 수 없다. quality는 이산 점수이고 다른 특성들이 함께 달라질 수 있다.

### 추가로 확인할 것
- wine type별 관계, 극단값의 기록 오류 여부, 점수별 표본 수와 비선형 패턴을 확인한다.


### Part 7 정리

> **발제자 포인트:** 숫자·분포·산점도·한계를 한 화면에 담는다. 인과 표현이 들어갔다면 고쳐 말하게 한다.
>
> **연결 문장:** 오늘의 세 문장으로 전체 흐름을 마무리하자.

---
## 마무리 [해석]

1. 데이터를 분석하기 전에 EDA로 데이터의 상태와 분포를 확인한다.
2. 변수 관계를 숫자로 확인할 때 공분산과 상관계수를 구분해서 사용한다.
3. 상관계수만 믿지 말고 실제 산점도와 데이터 분포를 함께 확인한다.

**PR 전 체크리스트**

- [ ] dtype과 결측치를 확인했는가
- [ ] 평균만 보고 분포를 판단하지 않았는가
- [ ] 공분산과 상관계수를 구분했는가
- [ ] 상관계수의 부호와 절댓값을 구분했는가
- [ ] r≈0을 "아무 관계 없음"으로 표현하지 않았는가
- [ ] 상관관계를 인과관계로 해석하지 않았는가
- [ ] 상관계수와 산점도를 함께 확인했는가
- [ ] 이상치 영향을 확인했는가
- [ ] 참여자용에 정답이 노출되지 않았는가
- [ ] 발제자용과 참여자용 셀 순서가 동일한가
- [ ] 개인 절대경로 없이 실행되는가
- [ ] 세 데이터 파일이 정상 로딩되는가
- [ ] 발제자용 전체 코드가 실제 실행되는가
